# 01. Fenwick Tree (Binary Indexed Tree)

A Fenwick tree stores **prefix sums with updates**. Prefix sum arrays answer queries in O(1) but updating one element costs O(n). A Fenwick tree makes **both update and query O(log n)**, with tiny code.

**Learning order:** REAL LIFE → IDEA → CODE → MUTABLE RANGE SUM → COUNT SMALLER AFTER SELF → INVERSIONS → WHEN TO USE → INTERVIEW THINKING

## 1. Real life

A shop keeps a running total of sales per day. Totals for "day 1 to day k" are needed often, but yesterday's number can be **corrected**. Recomputing a prefix array after every correction is slow. Fenwick tree handles both quickly.

**Recognition clue:** "range sum with updates", "count how many elements before/after are smaller", "dynamic prefix counts".

## 2. The idea

Every integer `i` is a sum of powers of two (binary). `tree[i]` stores the sum of a **block of length `lowbit(i)`** ending at `i`, where `lowbit(i) = i & -i`.

- **Query(i)** (prefix sum of `1..i`): add `tree[i]`, then jump to `i - lowbit(i)`, repeat until 0. At most `log n` jumps.
- **Update(i, delta)**: add `delta` to `tree[i]`, then jump to `i + lowbit(i)`, repeat until `> n`.

Indexes are **1-based** (index 0 would loop forever since `0 & -0 = 0`).

In [1]:
class FenwickTree:
    def __init__(self, n):
        self.n = n
        self.tree = [0] * (n + 1)

    def update(self, i, delta):                 # i is 1-based
        while i <= self.n:
            self.tree[i] += delta
            i += i & -i

    def query(self, i):                         # sum of 1..i
        total = 0
        while i > 0:
            total += self.tree[i]
            i -= i & -i
        return total

    def range_sum(self, l, r):                  # sum of l..r (1-based, inclusive)
        return self.query(r) - self.query(l - 1)

ft = FenwickTree(5)
for i, v in enumerate([1, 3, 5, 7, 9], start=1):
    ft.update(i, v)
print(ft.query(3), ft.range_sum(2, 4))   # 9 15
ft.update(3, 10)                         # element 3 becomes 15
print(ft.range_sum(2, 4))                # 25

9 15
25


## 3. Range Sum Query: Mutable

`update(index, val)` sets a value, `sum_range(l, r)` returns the sum. Store the current array to compute the **delta** `val - old`.

In [2]:
class NumArray:
    def __init__(self, nums):
        self.nums = nums[:]
        self.ft = FenwickTree(len(nums))
        for i, v in enumerate(nums, start=1):
            self.ft.update(i, v)

    def update(self, index, val):
        self.ft.update(index + 1, val - self.nums[index])
        self.nums[index] = val

    def sum_range(self, left, right):
        return self.ft.range_sum(left + 1, right + 1)

na = NumArray([1, 3, 5])
print(na.sum_range(0, 2))   # 9
na.update(1, 2)
print(na.sum_range(0, 2))   # 8

9
8


## 4. Count of Smaller Numbers After Self (hard, classic)

For each element, how many elements to its **right** are smaller?

1. **Coordinate compression:** replace values by their rank `1..m` (so the tree size is small).
2. Go from **right to left**. For `x`, the answer is `query(rank(x) - 1)` = how many smaller values have already been inserted.
3. Then `update(rank(x), 1)`.

O(n log n).

In [3]:
def count_smaller(nums):
    rank = {v: i + 1 for i, v in enumerate(sorted(set(nums)))}
    ft = FenwickTree(len(rank))
    res = []
    for x in reversed(nums):
        res.append(ft.query(rank[x] - 1))
        ft.update(rank[x], 1)
    return res[::-1]

print(count_smaller([5, 2, 6, 1]))   # [2, 1, 1, 0]

[2, 1, 1, 0]


## 5. Counting inversions with a Fenwick tree

Go left to right. Elements already inserted that are **greater** than `x` = `inserted so far - query(rank(x))`. Same O(n log n) as merge sort, but simpler to adapt (for example Reverse Pairs, Global and Local Inversions).

In [4]:
def count_inversions(nums):
    rank = {v: i + 1 for i, v in enumerate(sorted(set(nums)))}
    ft = FenwickTree(len(rank))
    inversions = 0
    for i, x in enumerate(nums):
        inversions += i - ft.query(rank[x])      # how many earlier values are > x
        ft.update(rank[x], 1)
    return inversions

print(count_inversions([2, 4, 1, 3, 5]), count_inversions([5, 4, 3, 2, 1]))   # 3 10

3 10


## 6. Fenwick vs Segment tree vs Prefix sum

| | Prefix array | Fenwick | Segment tree |
|---|---|---|---|
| point update | O(n) | **O(log n)** | O(log n) |
| prefix / range sum | **O(1)** | O(log n) | O(log n) |
| min / max queries | no | hard (needs invertible op) | **yes** |
| range updates | no | with tricks | with lazy propagation |
| code size | tiny | **tiny** | longer |

Fenwick works for operations that can be **undone** (sum, xor, count). For min/max use a segment tree.

## 7. Interview thinking

**Say:** "A prefix array would make each update O(n). A Fenwick tree gives O(log n) update and query, with O(n) memory. For 'count smaller to the right' I compress values and query counts of smaller ranks."

**Common mistakes:** 0-based indexes, forgetting coordinate compression (values up to 10⁹), computing the delta wrongly in updates, mixing up query of `rank - 1` and `rank`.

## Quick Revision

- Fenwick: 1-based, `i & -i`. Query goes down (`i -= lowbit`), update goes up (`i += lowbit`).
- O(log n) update and prefix query, O(n) space.
- Range sum = `query(r) - query(l-1)`.
- Count smaller after self / inversions: compress, iterate, query ranks.
- Only for invertible operations (sum, xor).